In [1]:
%cd ../../

/Users/hoangle/Projects/untangling-people/fwo_models


In [10]:
from pathlib import Path

import polars as pl
import yaml

from src import Data

# Load data

## Load data dimensions and POS

In [3]:
data = Data()

### Process meals' info

In [4]:
meals = (
    data.dim_meals
    .filter(pl.col('restaurant').is_not_null())

    .select('meal_id', 'meal_type', 'restaurant')
    .explode('restaurant')
)

meals.head()

meal_id,meal_type,restaurant
i64,str,str
9017,"""vegan""","""che"""
9017,"""vegan""","""exa"""
9017,"""vegan""","""vik"""
7010,"""vegetarian""","""che"""
7010,"""vegetarian""","""exa"""


### Process POS

In [5]:
CUTOFF_DATE = pl.lit("2024-09-01", dtype=pl.Date)

meals_pos = (
    data.pos
    
    # Keep entries in train split
    .filter(pl.col('date') < CUTOFF_DATE)

    
    # Get representative POS for each meal
    .group_by('meal_id', 'meal_type', 'restaurant')
    .agg(pl.col('pcs').median())
)


meals_pos.head()

meal_id,meal_type,restaurant,pcs
i64,str,str,f64
7589,"""vegan""","""che""",238.0
90000147,"""vegan""","""exa""",133.5
90000104,"""vegetarian""","""vik""",15.0
90000018,"""vegan""","""che""",223.5
90000010,"""chicken""","""vik""",117.5


### Process embeddings

In [6]:
embeddings = (
    data.dim_embds
    .select(
        'meal_id',
        pl.col('embedding').list.to_array(1024)
    )
)
embeddings.head()

meal_id,embedding
i64,"array[f64, 1024]"
9017,"[0.009898, -0.026522, … -0.002193]"
7201,"[0.056879, -0.003364, … 0.037541]"
9032,"[-0.001003, 0.024548, … 0.020614]"
9102,"[0.093415, 0.032748, … 0.028103]"
7010,"[0.011746, -0.038603, … 0.036116]"


### Combine POS into meal info

In [7]:
meals = (
    meals

    # Add POS data
    .join(meals_pos, on=['meal_id', 'meal_type', 'restaurant'], how='full')
    .filter(pl.col('meal_id').is_not_null())
    .select(
        'meal_id', 'meal_type', 'restaurant',
        pl.col('pcs').fill_null(-1)
    )

    # Add embedding info
    .join(embeddings, on='meal_id', how='left')
)


meals.head()

meal_id,meal_type,restaurant,pcs,embedding
i64,str,str,f64,"array[f64, 1024]"
9017,"""vegan""","""che""",-1.0,"[0.009898, -0.026522, … -0.002193]"
9017,"""vegan""","""exa""",-1.0,"[0.009898, -0.026522, … -0.002193]"
9017,"""vegan""","""vik""",-1.0,"[0.009898, -0.026522, … -0.002193]"
7010,"""vegetarian""","""che""",142.0,"[0.011746, -0.038603, … 0.036116]"
7010,"""vegetarian""","""exa""",71.0,"[0.011746, -0.038603, … 0.036116]"


# Save processed data

In [8]:
with open('src/embedding_tuning/conf.yaml') as file:
    conf = yaml.safe_load(file)

In [11]:
path = Path(conf['PATHS']['meals'])
path.parent.mkdir(exist_ok=True, parents=True)

meals.write_parquet(path)